[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hien078/applied-mathematics-foundation/blob/master/numerical_methods/05_numerical_differentiation/exercises.ipynb)

# Exercises — Topic 05: Numerical Differentiation

20 fully solved problems in 4 levels: concept checks, foundational computations and derivations, AI/ML and physics applications, and challenge proofs.

## Level 0 — Concept Check

### Problem L0.1: Why the limit does not exist numerically

Explain, from first principles, why $\frac{f(x+h)-f(x)}{h}$ stops improving as $h \to 0$ in floating point, and sketch the shape of the error curve on log–log axes.

**Solution.**

Two errors act in opposition.

**Truncation.** Taylor's theorem gives $\frac{f(x+h)-f(x)}{h} = f'(x) + \frac{h}{2}f''(\xi)$, so this contribution is $\approx \frac{h}{2}M_2$ and shrinks linearly in $h$.

**Roundoff.** Each evaluation returns $f(x)(1+\theta)$ with $\lvert \theta \rvert \le \varepsilon_M$, an absolute error $\approx \varepsilon_M \lvert f \rvert$. As $h \to 0$ the two values $f(x+h)$ and $f(x)$ agree in more and more leading digits, so their difference — of true size $\approx h\lvert f' \rvert$ — is contaminated by an absolute error of $2\varepsilon_M \lvert f\rvert$ that does **not** shrink. Dividing by $h$ amplifies it to $\frac{2\varepsilon_M \lvert f \rvert}{h}$, which *grows* as $h \to 0$.

**Total.**

$$
E(h) \approx \frac{h}{2}M_2 + \frac{2\varepsilon_M \lvert f \rvert}{h}.
$$

On log–log axes, $\log E$ versus $\log h$ is a **V**: a straight line of slope $+1$ for large $h$ (truncation regime), a straight line of slope $-1$ for small $h$ (roundoff regime), with a rounded minimum in between at $h^{\ast} \approx 2\sqrt{\varepsilon_M} \approx 3\times10^{-8}$ where the two contributions balance. Measured on $f = \sin$, $x = 1$: error $4.2\times10^{-4}$ at $h = 10^{-3}$, minimum $3.0\times10^{-9}$ at $h = 10^{-8}$, then back up to $5.4\times10^{-1}$ at $h = 10^{-16}$ (where $x + h = x$ exactly and the quotient is $0/h$-style garbage).

$$
\boxed{E(h) \approx \tfrac{h}{2}M_2 + \tfrac{2\varepsilon_M \lvert f \rvert}{h}: \text{ V-shaped, minimum at } h^{\ast}\sim\sqrt{\varepsilon_M}, \text{ best error } \sim\sqrt{\varepsilon_M}}
$$

*Key takeaway:* "Take $h$ as small as possible" is the single most common and most costly mistake in numerical differentiation.

### Problem L0.2: Order of accuracy is not a constant factor

At $h = 10^{-4}$, compare the truncation errors of the forward and central difference for a function with $M_2 = M_3 = 1$. By what factor do they differ, and what does that imply about the word "twice as accurate"?

**Solution.**

$$
E_{\text{fwd}} \approx \frac{h}{2}M_2 = \frac{10^{-4}}{2} = 5\times10^{-5}, \qquad E_{\text{cen}} \approx \frac{h^{2}}{6}M_3 = \frac{10^{-8}}{6} = 1.67\times10^{-9}.
$$

Ratio:

$$
\frac{E_{\text{fwd}}}{E_{\text{cen}}} = \frac{h/2}{h^2/6} = \frac{3}{h} = 3\times10^{4}.
$$

Not "twice"; **thirty thousand times**. And the ratio is not fixed — it grows as $3/h$, so at $h = 10^{-6}$ it is $3\times10^{6}$.

The cost, meanwhile, is only one extra function evaluation per point (two instead of one, or the same total if $f(x)$ is needed anyway). This is the best accuracy-per-flop bargain in the entire subject.

The general statement: an order-$p$ formula has error $C h^{p}$, so comparing orders $p$ and $q$ gives a ratio $\propto h^{\,q-p}$ that diverges as $h \to 0$. Constants matter only when the orders are equal.

$$
\boxed{E_{\text{fwd}}/E_{\text{cen}} = 3/h = 3\times10^{4} \text{ at } h = 10^{-4}, \text{ and it grows as } h \text{ shrinks}}
$$

*Key takeaway:* Always compare methods by *order* first; error constants are a tiebreaker between equal orders, never a substitute.

### Problem L0.3: Diagnosing a regime from data

A colleague reports these absolute errors for a derivative estimate: $h = 10^{-2}$: $9.0\times10^{-6}$; $h = 10^{-3}$: $9.0\times10^{-8}$; $h = 10^{-4}$: $9.0\times10^{-10}$; $h = 10^{-6}$: $2.8\times10^{-11}$; $h = 10^{-9}$: $3.0\times10^{-9}$. Identify the method's order, the regime at each $h$, and the approximately optimal step.

**Solution.**

**Order from the left branch.** From $h = 10^{-2}$ to $10^{-3}$ the error falls by $10^{2}$ for a $10\times$ reduction in $h$; likewise from $10^{-3}$ to $10^{-4}$. So $E \propto h^{2}$: the method is **second order** — a central difference. Fitting, $E_t \approx 9.0\times10^{-2}\,h^{2}$, i.e. $\frac{M_3}{6} = 0.09$, so $M_3 \approx 0.54$ (consistent with $f = \sin$ at $x = 1$, where $\lvert f''' \rvert = \cos 1 = 0.540$).

**Regimes.**
- $h = 10^{-2}, 10^{-3}, 10^{-4}$: clean $h^2$ scaling — **truncation dominated**.
- $h = 10^{-6}$: the extrapolated truncation error would be $9\times10^{-14}$, but the observed error is $2.8\times10^{-11}$ — three orders larger. **Roundoff dominated**.
- $h = 10^{-9}$: error $3.0\times10^{-9}$, consistent with $\varepsilon_M \lvert f \rvert / h = 2.2\times10^{-16}\times0.84/10^{-9} \approx 1.9\times10^{-7}$ order-of-magnitude bound; deep in the **roundoff branch**, error growing as $1/h$.

**Optimal step.** The minimum sits where the branches cross:

$$
0.09\,h^{2} = \frac{\varepsilon_M \lvert f \rvert}{h} \implies h^{3} = \frac{1.87\times10^{-16}}{0.09} \implies h^{\ast} \approx 1.3\times10^{-5},
$$

matching the theoretical $h^{\ast} = (3\varepsilon_M \lvert f\rvert / M_3)^{1/3} = 1.0\times10^{-5}$ and the measured minimum near $h = 10^{-5}$ with error $\approx 1.1\times10^{-11}$.

$$
\boxed{\text{Second order; truncation for } h \ge 10^{-4}, \text{ roundoff for } h \le 10^{-6}; \; h^{\ast} \approx 10^{-5}}
$$

*Key takeaway:* Reading the log–log slope of an error sweep tells you the order ($+p$ branch) and immediately locates the optimal step at the vertex of the V.

### Problem L0.4: Choosing a differentiation method

Select a method for each: (a) training a neural network in PyTorch; (b) sensitivity of a closed-source Fortran CFD binary; (c) sensitivity of the same solver, but you have the source and it uses only arithmetic and analytic intrinsics; (d) verifying a hand-written backward pass for a new layer.

**Solution.**

(a) **Reverse-mode automatic differentiation (backpropagation).** The gradient has $n \sim 10^{9}$ components; reverse mode delivers all of them exactly at $O(1)$ function evaluations, while central differences would need $2\times10^{9}$ forward passes and be accurate only to $\varepsilon_M^{2/3}$.

(b) **Central differences with $h \approx \varepsilon_M^{1/3}\max(\lvert x\rvert, 1)$** — a black box permits nothing else. Expect $\sim 11$ correct digits at best (fewer, since a CFD solver converged to $10^{-8}$ has noise $\eta = 10^{-8}$, giving $h^{\ast} = (3\times10^{-8})^{1/3} \approx 3\times10^{-3}$ and accuracy only $\eta^{2/3} \approx 5\times10^{-6}$). Cost: $2n$ solver runs.

(c) **Complex-step differentiation** (or an AD tool). Recompile with complex arithmetic, evaluate $\operatorname{Im} f(x+ih)/h$ with $h = 10^{-20}$: machine-precision derivatives, one complex run per input direction, no step-size tuning. This is precisely the situation Squire & Trapp and Martins et al. targeted. Watch for `abs`, `max`, and comparison branches, which break analyticity.

(d) **Central differences in float64** — deliberately a method sharing no code with the analytic gradient. `torch.autograd.gradcheck` with `dtype=torch.float64`, $h = 10^{-6}$, evaluated away from kinks; accept relative error below $\sim10^{-6}$.

$$
\boxed{\text{(a) reverse-mode AD, (b) central differences, (c) complex step, (d) central differences in float64}}
$$

*Key takeaway:* The choice is set by three questions — do you have the source, is $f$ analytic, and how many derivative components do you need?

## Level 1 — Foundation

### Problem L1.1: Three stencils on $e^{x}$

For $f(x) = e^{x}$ at $x = 0$ with $h = 0.1$, compute the forward difference, central difference, and second-derivative stencil. Compare each to its predicted truncation error.

**Solution.**

With $e^{0.1} = 1.1051709181$ and $e^{-0.1} = 0.9048374180$, and $f'(0) = f''(0) = 1$:

**Forward.**

$$
\frac{e^{0.1} - 1}{0.1} = \frac{0.1051709181}{0.1} = 1.051709181, \qquad \text{error } = 5.1709\times10^{-2}.
$$

Predicted: $\frac{h}{2}f''(\xi) = 0.05\,e^{\xi}$ with $\xi \in (0, 0.1)$, i.e. between $0.05$ and $0.05526$ — and $0.05171$ lies in that band. ✔

**Central.**

$$
\frac{e^{0.1} - e^{-0.1}}{0.2} = \frac{0.2003335000}{0.2} = 1.001667500, \qquad \text{error } = 1.6675\times10^{-3}.
$$

Predicted: $\frac{h^2}{6}f'''(\xi) = \frac{0.01}{6}e^{\xi} \approx 1.6667\times10^{-3}$. Agreement to 4 digits. ✔ Note the central error is **31 times smaller** than the forward error at the same $h$, using one extra evaluation.

**Second derivative.**

$$
\frac{e^{0.1} - 2 + e^{-0.1}}{0.01} = \frac{0.0100083361}{0.01} = 1.00083336, \qquad \text{error } = 8.3336\times10^{-4}.
$$

Predicted: $\frac{h^2}{12}f^{(4)}(\xi) = \frac{0.01}{12} = 8.3333\times10^{-4}$. Agreement to 4 digits. ✔

$$
\boxed{1.051709 \ (5.17\times10^{-2}), \quad 1.001668 \ (1.67\times10^{-3}), \quad 1.000833 \ (8.33\times10^{-4})}
$$

*Key takeaway:* At usable step sizes the Taylor error terms are not bounds but accurate *predictions* — you can budget your error before running anything.

### Problem L1.2: Computing the optimal step

For $f = \sin$ at $x = 1$ in IEEE double precision ($\varepsilon_M = 2.220\times10^{-16}$), predict $h^{\ast}$ and the best achievable error for the forward and central differences, then compare with the measured sweep.

**Solution.**

Here $\lvert f \rvert = \sin 1 = 0.841471$, $M_2 = \lvert{-\sin 1}\rvert = 0.841471$, $M_3 = \lvert{-\cos 1}\rvert = 0.540302$, and $\eta = \varepsilon_M$.

**Forward.**

$$
h^{\ast} = 2\sqrt{\frac{\varepsilon_M \lvert f \rvert}{M_2}} = 2\sqrt{\varepsilon_M} = 2(1.4901\times10^{-8}) = 2.98\times10^{-8},
$$

$$
E(h^{\ast}) = 2\sqrt{\varepsilon_M \lvert f\rvert M_2} = 2\,\lvert f\rvert \sqrt{\varepsilon_M} = 2(0.841471)(1.4901\times10^{-8}) = 2.51\times10^{-8}.
$$

**Central.**

$$
h^{\ast} = \left( \frac{3\varepsilon_M \lvert f \rvert}{M_3} \right)^{1/3} = \left( \frac{3(2.220\times10^{-16})(0.841471)}{0.540302} \right)^{1/3} = (1.0375\times10^{-15})^{1/3} = 1.012\times10^{-5},
$$

$$
E(h^{\ast}) = \frac{3^{2/3}}{2}M_3^{1/3}(\varepsilon_M \lvert f\rvert)^{2/3} = 1.0400 \times 0.8144 \times 3.267\times10^{-11} = 2.77\times10^{-11}.
$$

**Measured sweep** (absolute errors):

| $h$ | $10^{-3}$ | $10^{-5}$ | $10^{-8}$ | $10^{-12}$ |
| :--- | :--- | :--- | :--- | :--- |
| forward | $4.21\times10^{-4}$ | $4.21\times10^{-6}$ | $2.97\times10^{-9}$ | $4.32\times10^{-5}$ |
| central | $9.01\times10^{-8}$ | $1.11\times10^{-11}$ | $2.58\times10^{-9}$ | $1.23\times10^{-5}$ |

The forward minimum is at $h \approx 10^{-8}$ with error $2.97\times10^{-9}$; the central minimum is at $h \approx 10^{-5}$ with error $1.11\times10^{-11}$. The predicted *locations* are right to within a factor of $3$, and the predicted *magnitudes* are pessimistic by about $8\times$ and $2.5\times$ — expected, since the roundoff bound assumes all errors align in the worst direction while in practice signs are effectively random and partially cancel.

$$
\boxed{h^{\ast}_{\text{fwd}} = 3.0\times10^{-8},\, E = 2.5\times10^{-8}; \quad h^{\ast}_{\text{cen}} = 1.0\times10^{-5},\, E = 2.8\times10^{-11}}
$$

*Key takeaway:* The formulas predict both where the minimum sits and how deep it is; use them to decide *before* the experiment whether finite differences can meet your accuracy target at all.

### Problem L1.3: A one-sided formula by undetermined coefficients

Derive the second-order one-sided formula for $f'(x)$ using $f(x), f(x+h), f(x+2h)$, and find its error constant. Why is it needed and what does it cost?

**Solution.**

**Ansatz.** $D_h f(x) = \frac{1}{h}\bigl[c_0 f(x) + c_1 f(x+h) + c_2 f(x+2h)\bigr]$, i.e. stencil offsets $a = (0, 1, 2)$, $k = 1$, $m = 3$.

**Conditions** $S_i = \sum_j c_j \frac{a_j^{\,i}}{i!} = \delta_{i,1}$ for $i = 0,1,2$:

$$
i=0:\ c_0 + c_1 + c_2 = 0, \qquad i=1:\ c_1 + 2c_2 = 1, \qquad i=2:\ \frac{c_1 + 4c_2}{2} = 0 .
$$

From the third, $c_1 = -4c_2$; substituting into the second, $-4c_2 + 2c_2 = 1 \Rightarrow c_2 = -\tfrac12$, hence $c_1 = 2$ and $c_0 = -\tfrac32$. Therefore

$$
f'(x) \approx \frac{-3f(x) + 4f(x+h) - f(x+2h)}{2h}.
$$

**Error constant.** The leading error is $h^{2}f'''(x)\,S_3$ with

$$
S_3 = \frac{1}{6}\Bigl[ 0 + 2(1)^3 - \tfrac12 (2)^3 \Bigr] = \frac{2 - 4}{6} = -\frac{1}{3},
$$

so

$$
D_h f(x) = f'(x) - \frac{h^{2}}{3}f'''(\xi) .
$$

**Why and at what cost.** At a domain boundary (the first grid point of a PDE mesh, the edge of a data table, or a variable at a box constraint) the central stencil would need a point outside the domain. The one-sided rule solves that — but its error constant $\tfrac13$ is **twice** the central rule's $\tfrac16$ at the same order, and it uses three evaluations instead of two. Its roundoff amplification is also larger ($\frac{(3+4+1)\eta\lvert f\rvert}{2h} = \frac{4\eta \lvert f\rvert}{h}$ versus $\frac{\eta\lvert f\rvert}{h}$), so its optimal step and floor are both worse.

$$
\boxed{f'(x) = \frac{-3f(x)+4f(x+h)-f(x+2h)}{2h} + \frac{h^{2}}{3}f'''(\xi)}
$$

*Key takeaway:* Undetermined coefficients generate any stencil mechanically — and simultaneously deliver its error constant, which is what tells you the price of one-sidedness.

### Problem L1.4: A Richardson table for the derivative

Estimate $f'(1)$ for $f = \sin$ using central differences at $h = 0.1, 0.05, 0.025$ and build two levels of Richardson extrapolation. Verify the observed orders. (Exact: $\cos 1 = 0.5403023058681398$.)

**Solution.**

**Column 0 — central differences $D(h) = \frac{\sin(1+h)-\sin(1-h)}{2h}$.**

| $h$ | $D(h)$ | error |
| :--- | :--- | :--- |
| $0.1$ | $0.5394022521697600$ | $-9.0005\times10^{-4}$ |
| $0.05$ | $0.5400772080464322$ | $-2.2510\times10^{-4}$ |
| $0.025$ | $0.5402460261367148$ | $-5.6280\times10^{-5}$ |

Error ratios $4.00$ and $4.00$ — confirming order $2$ ($h \to h/2$ divides the error by $2^{2}$).

**Column 1 — one extrapolation, $R_1 = \frac{4D(h/2) - D(h)}{3}$.**

$$
R_1(0.1) = \frac{4(0.5400772080464322) - 0.5394022521697600}{3} = 0.5403021933386563, \quad \text{error } -1.1253\times10^{-7},
$$

$$
R_1(0.05) = \frac{4(0.5402460261367148) - 0.5400772080464322}{3} = 0.5403022988334757, \quad \text{error } -7.0347\times10^{-9}.
$$

Ratio $16.0$ — order $4$, exactly as predicted since the central expansion has only even powers ($p = 2$, $q = 2$).

**Column 2 — second extrapolation, $R_2 = \frac{16 R_1(h/2) - R_1(h)}{15}$.**

$$
R_2 = \frac{16(0.5403022988334757) - 0.5403021933386563}{15} = 0.5403023058664637, \quad \text{error } -1.68\times10^{-12}.
$$

Order $6$: the error dropped by a further factor of $4200$.

**Accounting.** Three function-pair evaluations (six values of $\sin$) produced $12$ correct digits — versus $6$ correct digits from the best possible plain central difference at its optimal step. Richardson converted a modest-$h$, cheap computation into near-machine accuracy without ever going near the roundoff cliff.

$$
\boxed{D(0.025) \text{ err } 5.6\times10^{-5} \;\to\; R_1 \text{ err } 7.0\times10^{-9} \;\to\; R_2 \text{ err } 1.7\times10^{-12}}
$$

*Key takeaway:* Extrapolate at moderate $h$ where the Taylor expansion is clean; halving $h$ and combining beats shrinking $h$ alone, and never touches the cancellation regime.

### Problem L1.5: Complex step versus central difference

For $f = \sin$ at $x = 1$, compute the complex-step derivative at $h = 10^{-3}$, $10^{-7}$, and $10^{-8}$, compare with the central difference, and explain the observed behaviour.

**Solution.**

**Formula.** $\sin(x + ih) = \sin x \cosh h + i\cos x \sinh h$, so

$$
\frac{\operatorname{Im}\sin(x+ih)}{h} = \cos x\,\frac{\sinh h}{h} = \cos x\left(1 + \frac{h^{2}}{6} + \frac{h^4}{120} + \cdots\right),
$$

confirming the general expansion $f' - \frac{h^2}{6}f'''$ (here $f''' = -\cos x$, so the sign works out to $+\frac{h^2}{6}\cos x$).

**Measured errors** (IEEE double):

| $h$ | complex step | central difference |
| :--- | :--- | :--- |
| $10^{-3}$ | $9.005\times10^{-8}$ | $9.005\times10^{-8}$ |
| $10^{-5}$ | $9.005\times10^{-12}$ | $1.11\times10^{-11}$ |
| $10^{-7}$ | $8.88\times10^{-16}$ | $1.94\times10^{-10}$ |
| $10^{-8}$ | $0$ (exact) | $2.58\times10^{-9}$ |
| $10^{-12}$ | $0$ (exact) | $1.23\times10^{-5}$ |

**Reading the table.**
- At $h = 10^{-3}$ both are purely truncation-limited and both give $\frac{h^2}{6}\lvert\cos 1\rvert = 9.0\times10^{-8}$ — identical, because both are second-order with the same error constant.
- Below $h \approx 10^{-5}$ they diverge completely: the central difference hits its roundoff floor and turns around, while the complex step keeps improving monotonically and reaches **exactly the correctly rounded value** for every $h \le 10^{-8}$.
- The reason is structural: $\operatorname{Im}\sin(x+ih) = \cos x \sinh h$ is *itself* $O(h)$, computed to full relative precision; nothing nearly equal is ever subtracted. Dividing by $h$ preserves relative accuracy, so the roundoff term is $O(\varepsilon_M \lvert f' \rvert)$ **independent of $h$**.

**The cost and the caveat.** One complex evaluation costs about 2–4 real ones. But $f$ must be genuinely analytic *as implemented*: `abs(z)`, `max`, sign tests, and any `real()` truncation break it silently. For $f(x) = \lvert x \rvert$ near $x = 1$, a naive complex `abs` returns $\sqrt{x^2+h^2}$ whose imaginary part is $0$ — the method reports $f' = 0$, confidently and wrongly.

$$
\boxed{\text{Complex step: error } \tfrac{h^2}{6}\lvert f'''\rvert + O(\varepsilon_M), \text{ monotone in } h, \text{ exact for } h \le 10^{-8}}
$$

*Key takeaway:* Removing the subtraction removes the entire roundoff branch of the V-curve — accuracy then costs nothing but analyticity.

### Problem L1.6: Reading orders off a log–log plot

You differentiate the same function with three methods and observe, over $h \in [10^{-3}, 10^{-1}]$, straight lines on a log–log error plot with slopes $1.00$, $2.00$, and $4.00$. Predict how each error changes when $h$ is halved, identify the methods, and estimate where each will bottom out in double precision.

**Solution.**

**Slope = order.** If $E = Ch^{p}$ then $\log E = \log C + p \log h$, so the slope of the line is exactly the order $p$.

**Halving $h$.** Error is multiplied by $2^{-p}$:

- $p = 1$: error $\times \tfrac12$ — one **forward or backward** difference.
- $p = 2$: error $\times \tfrac14$ — a **central** difference (or a one-sided 3-point rule, or the second-derivative stencil).
- $p = 4$: error $\times \tfrac{1}{16}$ — a **five-point central** stencil, or one Richardson extrapolation of a central difference.

**Where each bottoms out.** Balancing $C_t h^{p}$ against $\varepsilon_M \lvert f\rvert / h$ gives $h^{\ast} \sim \varepsilon_M^{1/(p+1)}$ and floor $E^{\ast} \sim \varepsilon_M^{p/(p+1)}$:

| $p$ | $h^{\ast} \sim \varepsilon_M^{1/(p+1)}$ | floor $\sim \varepsilon_M^{p/(p+1)}$ | correct digits |
| :--- | :--- | :--- | :--- |
| 1 | $1.5\times10^{-8}$ | $1.5\times10^{-8}$ | $\approx 8$ |
| 2 | $6.1\times10^{-6}$ | $3.7\times10^{-11}$ | $\approx 10$–$11$ |
| 4 | $7.4\times10^{-4}$ | $3.0\times10^{-13}$ | $\approx 12$–$13$ |

Note the diminishing returns: going $p = 1 \to 2$ buys three digits, $2 \to 4$ buys only two more, and the limit as $p \to \infty$ is $\varepsilon_M$ itself. Meanwhile each order increase needs more evaluations and pushes the optimal step *up*, which is good for conditioning but requires the function to be smooth over a wider window.

$$
\boxed{\text{slope} = p; \text{ halving } h \text{ scales error by } 2^{-p}; \text{ floor} \sim \varepsilon_M^{p/(p+1)}}
$$

*Key takeaway:* A single log–log error sweep is the complete diagnostic: the left slope gives the order, the vertex gives the optimal step, and the right slope ($-1$, or $-2$ for second derivatives) confirms the roundoff mechanism.

## Level 2 — Applications in AI/ML & Physics

### Problem L2.1: Why gradient checking demands float64

`torch.autograd.gradcheck` refuses to run in float32 by default. Quantify the best possible agreement between analytic and central-difference gradients in float32 and in float64, and explain the practical consequence.

**Solution.**

**Machine epsilons** (the `np.finfo` convention, $\varepsilon_M = 2^{1-t}$ for a $t$-bit significand). float32: $\varepsilon_M = 2^{-23} = 1.192\times10^{-7}$. float64: $\varepsilon_M = 2^{-52} = 2.220\times10^{-16}$.

**Best achievable central-difference accuracy** is $\sim \varepsilon_M^{2/3}$:

$$
\text{float32: } (1.192\times10^{-7})^{2/3} = 2.42\times10^{-5}, \qquad \text{float64: } (2.220\times10^{-16})^{2/3} = 3.67\times10^{-11}.
$$

**Consequence.** `gradcheck` defaults are roughly `atol=1e-5`, `rtol=1e-3`. In float32 the *floor* of the comparison, $2.4\times10^{-5}$, already exceeds `atol` — so a **perfectly correct** backward pass fails the test. Worse, in float32 the optimal step is $\varepsilon_M^{1/3} = 4.9\times10^{-3}$, uncomfortably large: over such a step a nonlinear loss is genuinely curved, so the truncation error is real too. The test becomes uninformative in both directions — it can fail correct code and pass buggy code whose error happens to be below $10^{-5}$.

In float64 the floor is $3.7\times10^{-11}$, six orders below the tolerance, so any discrepancy above $10^{-6}$ is a genuine bug.

**Practical recipe.**

```python
import torch
layer = MyLayer().double()                       # float64 parameters
x = torch.randn(4, 8, dtype=torch.float64, requires_grad=True)
x = x + 0.1 * torch.sign(x)                      # push away from kinks at 0
assert torch.autograd.gradcheck(layer, (x,), eps=1e-6, atol=1e-7, rtol=1e-4)
```

Also: use a *fixed* random seed and disable dropout/batch-norm updates, or the two evaluations $f(x+h)$ and $f(x-h)$ sample different functions and the "noise" $\eta$ becomes $O(1)$ rather than $\varepsilon_M$.

$$
\boxed{\text{float32 floor } 2.4\times10^{-5} \gt \texttt{atol}; \text{ float64 floor } 3.7\times10^{-11} \ll \texttt{atol}}
$$

*Key takeaway:* Gradient checking is an application of the optimal-step theory — the precision of the *comparison method* sets the smallest bug you can detect.

### Problem L2.2: Matrix-free Hessian-vector products

A model has $n = 10^{8}$ parameters. Explain why the Hessian cannot be formed, derive the finite-difference Hessian-vector product with its step size, and give its cost and accuracy against the exact double-backward alternative.

**Solution.**

**Why not form $H$.** The Hessian has $n^{2} = 10^{16}$ entries; at 4 bytes each that is $4\times10^{16}$ bytes $= 40$ petabytes. Even one row is $400$ MB. Any algorithm needing $H$ explicitly is dead on arrival; Newton-CG, Lanczos eigenvalue estimation, trust-region solvers, and influence functions are designed to need only the *action* $v \mapsto Hv$.

**Derivation.** $Hv$ is the directional derivative of the gradient:

$$
H(x)v = \left. \frac{d}{dt}\nabla f(x + tv) \right|_{t=0}.
$$

Applying the central difference in the scalar variable $t$:

$$
Hv \approx \frac{\nabla f(x + hv) - \nabla f(x - hv)}{2h} + \frac{h^{2}}{6}\left. \frac{d^{3}}{dt^{3}}\nabla f(x+tv)\right|_{t=\xi} .
$$

**Step size.** Applying Theorem 5 in the variable $t$, with the effective function magnitude $\lVert \nabla f \rVert$ and effective scale $\lVert x \rVert / \lVert v \rVert$:

$$
h = \frac{\sqrt{\varepsilon_M}\,\bigl(1 + \lVert x \rVert_2\bigr)}{\lVert v \rVert_2}
$$

is the standard choice (used in SciPy and in Newton-CG implementations). The $(1 + \lVert x\rVert)$ makes the perturbation relatively sized; dividing by $\lVert v \rVert$ makes it invariant to the scaling of $v$, which matters because CG feeds in vectors of wildly varying norm.

**Cost and accuracy.**

| Method | Cost | Accuracy | Memory |
| :--- | :--- | :--- | :--- |
| FD of gradients (central) | 2 gradient evaluations | $O(\varepsilon_M^{2/3})$ relative | none extra |
| FD of gradients (forward) | 1 extra gradient evaluation | $O(\sqrt{\varepsilon_M})$ relative | none extra |
| Double-backward (`torch.autograd.grad` twice) | $\approx 2$–$4\times$ one gradient | machine precision | stores the forward graph twice |
| Forward-over-reverse (`jax.jvp` of `jax.grad`) | $\approx 2\times$ one gradient | machine precision | modest |

**The stochastic trap.** With minibatch gradients, $\nabla f$ is a random variable whose fluctuation is $O(1)$ relative to the signal — the noise level is $\eta \sim 10^{-1}$, not $\varepsilon_M$. Then $h^{\ast} = (3\eta)^{1/3} \approx 0.67$, an absurd step, and the estimate is worthless. **The same minibatch and the same dropout mask must be used for both evaluations**; with that fixed, the noise reverts to $\varepsilon_M$ and the analysis above applies.

$$
\boxed{Hv \approx \frac{\nabla f(x+hv) - \nabla f(x-hv)}{2h}, \quad h = \frac{\sqrt{\varepsilon_M}(1+\lVert x\rVert)}{\lVert v \rVert}}
$$

*Key takeaway:* Second-order information at first-order cost — but only if the "function" being differenced is deterministic; freeze every source of randomness first.

### Problem L2.3: Gradient checking at a ReLU kink

A gradcheck of a ReLU network fails with a discrepancy of $0.45$ at one input coordinate. The analytic backward pass is correct. Diagnose the failure quantitatively and give the fix.

**Solution.**

**The mechanism.** ReLU is $\phi(x) = \max(0, x)$, differentiable everywhere except $x = 0$. PyTorch's convention is $\phi'(0) = 0$. Suppose a pre-activation sits at $x_0 = -10^{-7}$ and gradcheck uses $h = 10^{-6}$. Then

$$
\phi(x_0 + h) = \phi(9\times10^{-7}) = 9\times10^{-7}, \qquad \phi(x_0 - h) = \phi(-1.1\times10^{-6}) = 0,
$$

so the central difference reports

$$
\frac{9\times10^{-7} - 0}{2\times10^{-6}} = 0.45,
$$

while the analytic derivative at $x_0 \lt 0$ is $0$. Discrepancy: exactly $0.45$. **The finite difference is the wrong one**: it computes the secant slope across the kink, which is a chord of a non-differentiable function, not a derivative.

**Generalization.** For $x_0 \in (-h, h)$ the central difference of ReLU returns $\frac{x_0 + h}{2h} \in (0,1)$ — any value in the subdifferential, chosen by where $x_0$ sits in the window. The same happens with `abs`, `max`/`min`, `clamp`, `sign`, `round`, hard attention, top-$k$ selection, and quantization; also with `sqrt` at $0$ and `log` near $0$, where the derivative is unbounded and Taylor's theorem has no finite $M_2$.

**Fixes, in order of preference.**
1. **Move the test point off the kink**: sample inputs and initialize weights so no pre-activation lies within $\sim100h$ of a kink; e.g. `x = x + 0.1 * torch.sign(x)`.
2. **Use a smooth surrogate for the test**: check the same code path with GELU/Softplus, which are analytic, then trust the ReLU path structurally.
3. **Shrink $h$** so the kink window $(-h, h)$ is unlikely to be hit — but do not go below $\varepsilon_M^{1/3}$, which reintroduces roundoff.
4. **Use `nondet_tol` / per-element inspection** to confirm the failures are isolated to near-kink coordinates rather than systematic.

**What is not a fix.** Loosening `atol` until the test passes: that hides genuine bugs whose signature is a small systematic bias across *all* coordinates, precisely the opposite of the isolated large spike a kink produces.

$$
\boxed{\text{Central difference of ReLU at } x_0 = -10^{-7},\, h = 10^{-6} \text{ returns } 0.45 \text{ instead of } 0}
$$

*Key takeaway:* Finite differences assume $f \in C^{2}$; at a kink the assumption fails and the *finite difference*, not the analytic gradient, is the thing that is wrong.

### Problem L2.4: Zeroth-order gradients — SPSA and evolution strategies

An RL policy is evaluated by a non-differentiable simulator with $n = 10^{4}$ parameters. Compare coordinate-wise finite differences with SPSA and with an evolution-strategies estimator: cost per gradient, bias, and variance.

**Solution.**

**(a) Coordinate-wise central differences.**

$$
\hat g_i = \frac{f(x + h e_i) - f(x - h e_i)}{2h}, \qquad i = 1, \ldots, n .
$$

Cost: $2n = 20{,}000$ simulator rollouts per gradient. Bias: $O(h^{2})$ per coordinate. Variance: small if the simulator is deterministic, catastrophic otherwise. Verdict: exact-ish but unaffordable.

**(b) SPSA (Simultaneous Perturbation Stochastic Approximation).** Draw $\Delta \in \{\pm1\}^{n}$ (Rademacher) and set

$$
\hat g_i = \frac{f(x + c\Delta) - f(x - c\Delta)}{2c\,\Delta_i}.
$$

Cost: **2 evaluations, independent of $n$** — a $10^{4}$-fold saving.

*Unbiasedness (to $O(c^2)$).* Expanding, $f(x \pm c\Delta) = f \pm c\,\Delta^{\mathsf T}\nabla f + \tfrac{c^2}{2}\Delta^{\mathsf T}\nabla^2 f \Delta \pm O(c^3)$, so

$$
\hat g_i = \frac{\Delta^{\mathsf T}\nabla f}{\Delta_i} + O(c^{2}) = \partial_i f + \sum_{j\neq i} \frac{\Delta_j}{\Delta_i}\partial_j f + O(c^{2}).
$$

Since $\Delta_j/\Delta_i = \pm1$ with mean $0$ for $j \neq i$ (independence and symmetry), $\mathbb{E}[\hat g_i] = \partial_i f + O(c^{2})$: **unbiased up to the same $O(c^2)$ truncation term as any central difference**.

*Variance.* The cross terms give $\operatorname{Var}(\hat g_i) \approx \sum_{j \neq i}(\partial_j f)^2 = \lVert \nabla f\rVert^2 - (\partial_i f)^2$, i.e. $O(\lVert\nabla f\rVert^2)$ — huge for a single sample, which is why SPSA is run inside a stochastic-approximation loop with decaying gains $a_k = a/(k+A)^{\alpha}$, $c_k = c/k^{\gamma}$ (classically $\alpha = 0.602$, $\gamma = 0.101$).

**(c) Evolution strategies (Gaussian smoothing).** With $\epsilon_k \sim \mathcal{N}(0, I)$,

$$
\hat g = \frac{1}{\sigma N}\sum_{k=1}^{N}\epsilon_k\, f(x + \sigma\epsilon_k) .
$$

This is an unbiased estimator of $\nabla f_\sigma$, the gradient of the **smoothed** objective $f_\sigma(x) = \mathbb{E}_{\epsilon}[f(x+\sigma\epsilon)]$ — so the bias relative to $\nabla f$ is $O(\sigma^2 \lVert \nabla^3 f\rVert)$, not zero, but the smoothing is often *desirable* (it flattens the reward landscape). Cost: $N$ evaluations, embarrassingly parallel across workers with only a random seed communicated (the trick behind Salimans et al. 2017 scaling ES to 1440 CPUs). Antithetic sampling ($\pm\epsilon_k$) halves the variance and cancels the odd bias term exactly.

**Choosing $c$ and $\sigma$.** These are the $h$ of Theorem 5, with $\eta$ = the simulator's stochastic noise, typically $10^{-2}$ or worse. Then $c^{\ast} = (3\eta)^{1/3} \approx 0.3$ — the perturbations must be **large**, which is exactly what ES practitioners find empirically ($\sigma \approx 0.02$–$0.1$ after normalization).

$$
\boxed{\text{SPSA: } 2 \text{ evaluations for any } n, \text{ bias } O(c^2), \text{ variance } O(\lVert \nabla f\rVert^{2})}
$$

*Key takeaway:* Random-direction finite differences trade variance for a dimension-independent cost — the same truncation/noise analysis applies, but with the simulator's noise $\eta$ in place of $\varepsilon_M$, forcing much larger steps.

### Problem L2.5: The FTCS heat equation and its stability limit

Discretize $u_t = \alpha u_{xx}$ with a forward difference in time and a central second difference in space. Derive the scheme, its truncation order, and the von Neumann stability condition. Interpret the condition physically.

**Solution.**

**Scheme.** With $u_j^{n} \approx u(x_j, t_n)$, $\Delta x = h$, $\Delta t = \tau$:

$$
\frac{u_j^{n+1} - u_j^{n}}{\tau} = \alpha\,\frac{u_{j+1}^{n} - 2u_j^{n} + u_{j-1}^{n}}{h^{2}} \implies u_j^{n+1} = u_j^{n} + r\bigl( u_{j+1}^{n} - 2u_j^{n} + u_{j-1}^{n} \bigr), \quad r = \frac{\alpha\tau}{h^{2}} .
$$

**Truncation order.** By Theorem 1 the time difference is $O(\tau)$ and by Theorem 3 the space difference is $O(h^{2})$, so the local truncation error is

$$
T_j^{n} = \frac{\tau}{2}u_{tt} - \frac{\alpha h^{2}}{12}u_{xxxx} + O(\tau^{2}, h^{4}) = O(\tau + h^{2}) .
$$

The scheme is **first order in time, second order in space** — which already tells you $\tau$ must be small relative to $h^{2}$ for the two errors to balance, foreshadowing the stability result.

**Von Neumann analysis.** Insert the Fourier mode $u_j^{n} = G^{n}e^{ikjh}$:

$$
G = 1 + r\bigl(e^{ikh} - 2 + e^{-ikh}\bigr) = 1 + r\bigl(2\cos kh - 2\bigr) = 1 - 4r\sin^{2}\!\frac{kh}{2},
$$

using $1 - \cos\theta = 2\sin^2(\theta/2)$. Stability requires $\lvert G \rvert \le 1$ for all $k$. Since $\sin^2 \in [0,1]$, the binding case is $\sin^2 = 1$ (the sawtooth mode $kh = \pi$):

$$
-1 \le 1 - 4r \le 1 \iff 0 \le r \le \frac{1}{2} \implies \boxed{\ \tau \le \frac{h^{2}}{2\alpha}\ }
$$

**Physical reading.** $h^{2}/\alpha$ is the diffusion time across one cell. The condition says: **a timestep may not exceed half the time heat needs to cross one grid cell.** Violating it lets the explicit update propagate information faster than diffusion physically can, and the sawtooth mode amplifies by $\lvert 1-4r\rvert \gt 1$ each step — the classic checkerboard blow-up.

**The practical sting.** Refining the mesh by $2\times$ forces the timestep down by $4\times$, so the total work rises by $8\times$ in 1-D (and $2^{d+2}$ in $d$ dimensions). This is why implicit schemes (backward Euler, Crank–Nicolson) — unconditionally stable at the cost of a tridiagonal solve per step, and Crank–Nicolson $O(\tau^2 + h^2)$ — dominate stiff diffusion problems.

*Key takeaway:* The stencils of this topic are the atoms of PDE discretization, and their truncation orders plus a Fourier-mode test give the complete accuracy-and-stability picture of a scheme.

### Problem L2.6: Differentiating a Monte Carlo objective

A variational objective is estimated by Monte Carlo with $10^{6}$ samples, giving relative noise $\eta \approx 10^{-3}$. You need $\partial \mathcal{L}/\partial\theta$ by finite differences. Choose the step, predict the achievable accuracy, and give two better alternatives.

**Solution.**

**Step size.** Theorem 5 with $\eta = 10^{-3}$ (not $\varepsilon_M$!), assuming $O(1)$ derivatives:

$$
h^{\ast}_{\text{cen}} = \left( 3\eta \right)^{1/3} = (3\times10^{-3})^{1/3} = 0.144, \qquad h^{\ast}_{\text{fwd}} = 2\sqrt{\eta} = 0.063 .
$$

These are **enormous** steps — 14% and 6% of the parameter scale. Over such a step the objective is genuinely curved, so the "truncation error" is a real modelling error, not a formality.

**Achievable accuracy.**

$$
E_{\text{cen}} \sim \eta^{2/3} = 10^{-2}, \qquad E_{\text{fwd}} \sim \sqrt{\eta} = 3\times10^{-2} .
$$

About $1$–$2$ correct digits in the gradient. If the optimizer needs the gradient direction to $1\%$, this is barely adequate; for anything finer, finite differencing this objective is hopeless. Using $h = 10^{-6}$ "because small is accurate" would give error $\approx \eta/h = 10^{3}$ — the estimate is a thousand times larger than the true gradient and points in a random direction.

**Better alternative 1 — common random numbers.** Use the *same* random seed (the same $10^{6}$ samples) for $\mathcal{L}(\theta+h)$ and $\mathcal{L}(\theta-h)$. The two estimates then share their sampling error, which cancels in the difference: the effective $\eta$ drops from $10^{-3}$ to near $\varepsilon_M$, restoring $h^{\ast} \approx 10^{-5}$ and $\sim10$-digit accuracy. This single change is usually worth more than any algorithmic sophistication.

**Better alternative 2 — differentiate the estimator, not the estimate.** Rewrite the objective so the randomness does not depend on $\theta$ and apply AD:
- **Reparameterization (pathwise) gradient**: for $z \sim \mathcal{N}(\mu, \sigma^2)$, write $z = \mu + \sigma\epsilon$ with $\epsilon\sim\mathcal{N}(0,1)$; then $\nabla_\theta \mathbb{E}[g(z)] = \mathbb{E}[\nabla_\theta g(\mu + \sigma\epsilon)]$, an unbiased, low-variance, machine-precision gradient — the engine of the VAE.
- **Score-function (REINFORCE) gradient**: $\nabla_\theta \mathbb{E}_{p_\theta}[g] = \mathbb{E}_{p_\theta}[g\,\nabla_\theta \log p_\theta]$, unbiased for discrete variables where reparameterization is unavailable, though with much higher variance (hence control variates and baselines).

$$
\boxed{\eta = 10^{-3} \Rightarrow h^{\ast} = 0.14, \text{ accuracy only } 10^{-2}; \text{ use common random numbers or reparameterization}}
$$

*Key takeaway:* The noise level $\eta$, not machine epsilon, sets the step and the accuracy ceiling — and the best move is almost always to reduce $\eta$ (shared seeds) or to eliminate differencing entirely (reparameterize + AD).

## Level 3 — Challenge

### Problem L3.1: The universal accuracy ceiling

Prove that any difference formula for $f^{(k)}$ of order $p$, using function values with relative noise $\eta$, has a best achievable error $\Theta\bigl(\eta^{\,p/(p+k)}\bigr)$ at $h^{\ast} = \Theta\bigl(\eta^{1/(p+k)}\bigr)$. Deduce the ceilings for $(k,p) = (1,1), (1,2), (1,4), (2,2)$.

**Solution.**

**Setup.** Let $D_h f(x) = \frac{1}{h^{k}}\sum_{j=1}^{m}c_j f(x + a_j h)$ have order $p$, so its truncation error is $C_t h^{p}$ with $C_t = \lvert f^{(k+p)}(x) S_{k+p}\rvert$ from Proof 5 of the notebook. Each computed value carries an absolute error at most $\eta\lvert f\rvert$, so the roundoff contribution is at most

$$
\frac{1}{h^{k}}\sum_j \lvert c_j\rvert \,\eta \lvert f \rvert = \frac{C_r\,\eta\lvert f\rvert}{h^{k}}, \qquad C_r = \sum_j \lvert c_j \rvert .
$$

Note $C_r \gt 0$ always, since $\sum_j c_j a_j^{k}/k! = 1$ forces some $c_j \neq 0$.

**Total error and its minimizer.**

$$
E(h) = C_t h^{p} + \frac{C_r \eta \lvert f\rvert}{h^{k}} .
$$

$E$ is strictly convex on $(0,\infty)$ (both terms are convex; the second strictly), $E(h)\to\infty$ at both ends, so the minimizer is unique and interior:

$$
E'(h) = p\,C_t h^{p-1} - \frac{k\,C_r \eta\lvert f\rvert}{h^{k+1}} = 0 \implies h^{\ast} = \left( \frac{k\,C_r\,\eta\lvert f\rvert}{p\,C_t} \right)^{\frac{1}{p+k}} = \Theta\bigl(\eta^{\frac{1}{p+k}}\bigr).
$$

Substituting back,

$$
E(h^{\ast}) = C_t \left(\frac{kC_r\eta\lvert f\rvert}{pC_t}\right)^{\frac{p}{p+k}} + C_r\eta\lvert f\rvert\left(\frac{pC_t}{kC_r\eta\lvert f\rvert}\right)^{\frac{k}{p+k}} = \underbrace{\left(1 + \frac{p}{k}\right)\left(\frac{k}{p}\right)^{\frac{p}{p+k}}}_{\text{constant}} C_t^{\frac{k}{p+k}}\bigl(C_r\eta\lvert f\rvert\bigr)^{\frac{p}{p+k}} ,
$$

so $E(h^{\ast}) = \Theta\bigl(\eta^{\frac{p}{p+k}}\bigr)$. $\blacksquare$

**The ceilings** ($\eta = \varepsilon_M = 1.11\times10^{-16}$):

| $k$ | $p$ | $h^{\ast} \sim \varepsilon_M^{1/(p+k)}$ | ceiling $\sim \varepsilon_M^{p/(p+k)}$ | digits |
| :--- | :--- | :--- | :--- | :--- |
| 1 | 1 (forward) | $\varepsilon_M^{1/2} = 1.1\times10^{-8}$ | $\varepsilon_M^{1/2} = 1.1\times10^{-8}$ | 8 |
| 1 | 2 (central) | $\varepsilon_M^{1/3} = 4.8\times10^{-6}$ | $\varepsilon_M^{2/3} = 2.3\times10^{-11}$ | 11 |
| 1 | 4 (5-point) | $\varepsilon_M^{1/5} = 6.4\times10^{-4}$ | $\varepsilon_M^{4/5} = 1.7\times10^{-13}$ | 13 |
| 2 | 2 ($f''$ central) | $\varepsilon_M^{1/4} = 1.0\times10^{-4}$ | $\varepsilon_M^{1/2} = 1.1\times10^{-8}$ | 8 |

**Three structural conclusions.**
1. **Higher derivatives are strictly harder.** For fixed $p$, increasing $k$ lowers the exponent $\frac{p}{p+k}$: the second derivative at order 2 is as bad as the first derivative at order 1. A fourth derivative at order 2 would cap out at $\varepsilon_M^{1/3} \approx 5\times10^{-6}$.
2. **Order helps with diminishing returns.** As $p\to\infty$ the ceiling tends to $\eta$ — full precision is approached but never reached, and each extra order costs evaluations and a larger $C_t$ (higher derivatives of $f$ must exist and be moderate).
3. **The step grows with order.** Higher-order formulas want *larger* $h$, which is numerically comfortable but demands smoothness over a wider window — the reason high-order stencils fail badly on functions with nearby singularities or noise.

$$
\boxed{h^{\ast} = \Theta\bigl(\eta^{1/(p+k)}\bigr), \qquad E(h^{\ast}) = \Theta\bigl(\eta^{\,p/(p+k)}\bigr)}
$$

*Key takeaway:* No amount of cleverness with real function evaluations escapes this ceiling — only changing the *information* used (complex step, AD) does.

### Problem L3.2: Richardson to arbitrary order, and why it stops

Prove that the Romberg-style recursion $T_{k,m} = \frac{4^{m}T_{k,m-1} - T_{k-1,m-1}}{4^{m}-1}$ applied to central differences yields order $2m+2$, and prove that the roundoff amplification grows with $m$, bounding the useful depth.

**Solution.**

**Part 1 — order $2m+2$ by induction on $m$.**

*Base ($m = 0$).* $T_{k,0} = D(h_k)$ with $h_k = h/2^{k}$, and by Proof 2 the central difference has the even expansion

$$
T_{k,0} = f'(x) + \sum_{i \ge 1} c_i h_k^{2i}, \qquad c_i = \frac{f^{(2i+1)}(x)}{(2i+1)!},
$$

so the error is $O(h_k^{2})$, i.e. order $2 = 2\cdot 0 + 2$. ✔

*Inductive step.* Assume $T_{k,m-1} = f'(x) + \sum_{i \ge m} c_i^{(m-1)} h_k^{2i}$ — leading order $2m$. Since $h_k = h_{k-1}/2$, we have $h_k^{2i} = 4^{-i}h_{k-1}^{2i}$, so

$$
4^{m}T_{k,m-1} - T_{k-1,m-1} = (4^{m}-1)f'(x) + \sum_{i\ge m} c^{(m-1)}_i h_{k-1}^{2i}\left( 4^{m}4^{-i} - 1 \right).
$$

At $i = m$ the bracket is $4^{m}4^{-m} - 1 = 0$: the leading term is annihilated **exactly**. Dividing by $4^{m}-1$,

$$
T_{k,m} = f'(x) + \sum_{i \ge m+1}\underbrace{c_i^{(m-1)}\frac{4^{m-i}-1}{4^{m}-1}}_{c_i^{(m)}} h_{k-1}^{2i} = f'(x) + O(h^{2m+2}). \qquad \blacksquare
$$

Column $m$ therefore has order $2m+2$: $2, 4, 6, 8, \ldots$ — confirmed numerically for $f = \sin$ at $x=1$, where the errors of columns $0,1,2$ at the finest $h$ are $5.6\times10^{-5}$, $7.0\times10^{-9}$, $1.7\times10^{-12}$, with observed ratios $4$, $16$, $\approx 4000$ across successive $h$-halvings.

**Part 2 — roundoff amplification grows with $m$.**

Let $\epsilon_{k,m}$ denote the roundoff error contaminating $T_{k,m}$. The recursion is affine, so

$$
\lvert \epsilon_{k,m}\rvert \le \frac{4^{m}\lvert \epsilon_{k,m-1}\rvert + \lvert\epsilon_{k-1,m-1}\rvert}{4^{m}-1} \le \frac{4^{m}+1}{4^{m}-1}\max\bigl(\lvert\epsilon_{k,m-1}\rvert, \lvert\epsilon_{k-1,m-1}\rvert\bigr) .
$$

The factor $\gamma_m = \frac{4^{m}+1}{4^{m}-1}$ is $\frac53$ at $m=1$, $\frac{17}{15}$ at $m=2$, $\frac{65}{63}$ at $m=3$ — so *this* amplification is mild and its product $\prod_m \gamma_m$ converges. The real growth comes from a different direction: **column $m$ requires $T_{k,0}$ down to $h_k = h/2^{k}$ with $k \ge m$**, and the base-column roundoff is

$$
\lvert \epsilon_{k,0}\rvert \approx \frac{\eta\lvert f\rvert}{h_k} = \frac{2^{k}\eta\lvert f\rvert}{h},
$$

which grows **geometrically in $k$**. Combining, the extrapolated error behaves like

$$
E_m(h) \approx \underbrace{C\,h^{2m+2}}_{\text{truncation}} + \underbrace{\frac{2^{m}\,\Gamma\,\eta\lvert f\rvert}{h}}_{\text{roundoff}}, \qquad \Gamma = \prod_{j\le m}\gamma_j = O(1),
$$

so each extra column multiplies the roundoff floor by about $2$ while the truncation term gains two orders. Setting $E_m'(h) = 0$ gives $h^{\ast}_m = \Theta\bigl((2^{m}\eta)^{1/(2m+3)}\bigr)$ and a floor $\Theta\bigl((2^{m}\eta)^{(2m+2)/(2m+3)}\bigr)$ — improving in $m$ but with the $2^{m}$ eventually winning.

**Empirical stopping rule.** In the measured sweep for $f = \sin$ at $x = 1$, one Richardson level is *better* than the plain central difference for $h \ge 10^{-4}$ but **worse** for $h \le 10^{-6}$ (e.g. at $h = 10^{-10}$: central $5.8\times10^{-8}$ versus extrapolated $1.4\times10^{-6}$). Practical algorithm: build the table from a moderate $h$ (say $0.1$), monitor $\lvert T_{k,m} - T_{k,m-1}\rvert$, and **stop as soon as it stops decreasing** — typically after 3–5 columns, yielding 12–14 digits.

$$
\boxed{\text{column } m \text{ has order } 2m+2; \text{ roundoff floor grows like } 2^{m}\eta/h \Rightarrow \text{stop at } 3\text{--}5 \text{ columns}}
$$

*Key takeaway:* Richardson buys orders of accuracy for free in exact arithmetic and at geometric roundoff cost in floating point — always extrapolate from *large* $h$, never from small.

### Problem L3.3: Existence, uniqueness, and the error constant of any stencil

Prove that for distinct offsets $a_1, \ldots, a_m$ and any $k \lt m$ there is a unique weight vector giving an order-$(m-k)$ approximation to $f^{(k)}$, derive the exact leading error constant, and apply the machinery to obtain the five-point second-derivative formula.

**Solution.**

**Step 1 — the linear system.** Substituting the Taylor expansion $f(x + a_j h) = \sum_{i\ge0}\frac{(a_jh)^i}{i!}f^{(i)}(x)$ into $D_h f = h^{-k}\sum_j c_j f(x+a_jh)$ and collecting powers:

$$
D_h f(x) = \sum_{i \ge 0} h^{\,i-k}f^{(i)}(x)\,S_i, \qquad S_i := \sum_{j=1}^{m} c_j \frac{a_j^{\,i}}{i!} .
$$

Requiring $D_h f = f^{(k)} + O(h^{m-k})$ forces $S_i = \delta_{i,k}$ for $i = 0,\ldots,m-1$: $m$ equations, $m$ unknowns.

**Step 2 — existence and uniqueness.** In matrix form $Vc = e_k$ where $V_{ij} = a_j^{\,i}/i!$ for $i = 0,\ldots,m-1$. Factoring the row scalings, $V = \operatorname{diag}(1, 1, \tfrac{1}{2!}, \ldots, \tfrac{1}{(m-1)!})\,W$ with $W_{ij} = a_j^{\,i}$ a **Vandermonde** matrix. Hence

$$
\det V = \left(\prod_{i=0}^{m-1}\frac{1}{i!}\right)\prod_{1 \le i \lt j \le m}(a_j - a_i) \neq 0
$$

precisely because the offsets are distinct. So $c = V^{-1}e_k$ exists and is unique. $\blacksquare$

**Step 3 — the leading error constant.** With $S_i = \delta_{i,k}$ for $i \lt m$, the first surviving term is $i = m$:

$$
D_h f(x) - f^{(k)}(x) = h^{\,m-k}f^{(m)}(x)\,S_m + O(h^{\,m-k+1}), \qquad S_m = \frac{1}{m!}\sum_{j} c_j a_j^{\,m},
$$

and by the same Rolle/mean-value argument used in Topic 04's error theorem the term can be written exactly as $h^{m-k}f^{(m)}(\xi)S_m$ for some $\xi$ in the stencil's span whenever the weights make the functional a divided difference of one sign. **Symmetry bonus:** if the stencil is symmetric ($\{a_j\} = \{-a_j\}$) then by parity $S_m = 0$ automatically whenever $m - k$ is odd, and the order is $m - k + 1$, not $m-k$. This is the general form of "central differences gain an order for free."

**Step 4 — the five-point second derivative.** Take $k = 2$, $a = (-2,-1,0,1,2)$, $m = 5$. Symmetry gives $c_{-1} = c_1$, $c_{-2} = c_2$, so the unknowns are $c_0, c_1, c_2$ and the odd conditions $S_1 = S_3 = 0$ hold automatically. The even conditions:

$$
S_0 = c_0 + 2c_1 + 2c_2 = 0, \quad S_2 = \frac{2c_1 + 8c_2}{2} = 1, \quad S_4 = \frac{2c_1 + 32c_2}{24} = 0 .
$$

From $S_4$: $c_1 = -16c_2$. Into $S_2$: $-16c_2 + 4c_2 = 1 \Rightarrow c_2 = -\tfrac{1}{12}$, $c_1 = \tfrac{16}{12} = \tfrac43$, and $c_0 = -2(\tfrac43) - 2(-\tfrac{1}{12}) = -\tfrac83 + \tfrac16 = -\tfrac{30}{12}$. Thus

$$
f''(x) \approx \frac{-f(x-2h) + 16f(x-h) - 30f(x) + 16f(x+h) - f(x+2h)}{12h^{2}} .
$$

Because the stencil is symmetric and $m - k = 3$ is odd, $S_5 = 0$ and the order is $4$, not $3$. The constant comes from $S_6$:

$$
S_6 = \frac{1}{720}\Bigl[ 2\cdot\tfrac{4}{3}(1)^6 + 2\cdot\left(-\tfrac1{12}\right)(2)^{6} \Bigr] = \frac{1}{720}\left[ \tfrac83 - \tfrac{128}{12} \right] = \frac{-8}{720} = -\frac{1}{90},
$$

giving the error $-\frac{h^{4}}{90}f^{(6)}(\xi)$.

$$
\boxed{f''(x) = \frac{-f_{-2} + 16f_{-1} - 30f_0 + 16f_1 - f_2}{12h^{2}} + \frac{h^{4}}{90}f^{(6)}(\xi)}
$$

*Key takeaway:* One Vandermonde system generates every finite-difference formula ever published — including its order, its error constant, and the parity argument that explains why symmetric stencils are one order better than the count suggests.

### Problem L3.4: Why complex step escapes the ceiling — and what replaces it

Problem L3.1 proved a ceiling of $\Theta(\eta^{p/(p+k)})$ for any real-evaluation difference formula. Explain rigorously why complex-step differentiation is not a counterexample, prove its error model, and identify the exact boundary of its applicability.

**Solution.**

**Why it is not a counterexample.** The ceiling theorem assumes the estimate is a *linear combination of real function values divided by $h^{k}$*, so that the independent rounding errors $\eta\lvert f\rvert$ in each value are divided by $h^{k}$ and amplified without bound. The complex-step estimate is **not** of that form: it uses $\operatorname{Im}f(x+ih)$, a quantity that is *already* proportional to $h$. The theorem's hypothesis — that the numerator is an $O(1)$-sized quantity whose rounding error is $O(\eta)$ — fails. Nothing is contradicted; a different class of information is used.

**Error model.** Write $\operatorname{Im}f(x+ih) = h f'(x) - \frac{h^{3}}{6}f'''(x) + O(h^{5})$. Floating-point evaluation returns

$$
\widetilde{\operatorname{Im}f(x+ih)} = \operatorname{Im}f(x+ih)\,(1+\theta), \qquad \lvert\theta\rvert \le c\,\varepsilon_M
$$

for a modest constant $c$ counting the operations — this is a **relative** bound, valid because no catastrophic cancellation occurs: every intermediate imaginary part is $O(h)$ and they combine without subtracting nearly equal $O(1)$ quantities. Dividing by the exactly representable $h$ preserves relative accuracy, so

$$
\widetilde{D^{\mathrm{CS}}_h f} = \left(f'(x) - \frac{h^{2}}{6}f'''(x) + O(h^4)\right)(1+\theta) \implies E(h) \le \frac{h^{2}}{6}\lvert f'''\rvert + c\,\varepsilon_M\lvert f'\rvert .
$$

This is **monotone decreasing in $h$** with an $h$-independent floor $c\varepsilon_M \lvert f'\rvert$ — no V shape, no optimal step, no trade-off. Taking $h = 10^{-20}$ makes the truncation term $10^{-40}$, and the result is the correctly rounded $f'(x)$. The measured data confirm this exactly for $f = \sin$ at $x = 1$: error $9.0\times10^{-8}$ at $h = 10^{-3}$, $8.9\times10^{-16}$ at $h = 10^{-7}$, and identically $0$ for all $h \le 10^{-8}$.

**The boundary of applicability — three hard limits.**

1. **Analyticity is required, not merely smoothness.** $f$ must satisfy the Cauchy–Riemann equations in a neighbourhood, and the *implementation* must be the analytic continuation. Every operation must be complex-correct: `abs(z)` must be $\sqrt{z^2}$-consistent rather than $\sqrt{\bar z z}$, comparisons must not branch on $\operatorname{Re}$ alone, and `real(z)`, `conj(z)`, `min`, `max`, `sign`, `floor` all destroy analyticity. Failure is **silent**: the method returns a plausible wrong number. (For $f(x) = \lvert x\rvert$ with a library `abs`, $\operatorname{Im}\lvert x+ih\rvert = 0$, so the reported derivative is $0$ everywhere.)

2. **Only first derivatives, only one direction at a time.** The real part $\operatorname{Re}f(x+ih) = f(x) - \frac{h^{2}}{2}f''(x)$ recovers $f''$ only by subtracting $f(x)$ — restoring cancellation and the $\varepsilon_M/h^{2}$ floor. Second derivatives need **multicomplex** or **hyper-dual** numbers, at which point one has re-derived forward-mode automatic differentiation with nilpotents ($\epsilon^2 = 0$) replacing $i$ ($i^2 = -1$). For a gradient in $n$ variables, $n$ separate complex runs are needed — the same $O(n)$ scaling as forward-mode AD, and $n$ times worse than reverse mode.

3. **Source access and recompilation.** The whole codebase must be re-typed to complex, roughly doubling memory and costing 2–4× in time. If you can do that, you can usually also apply an AD tool, which gives exact derivatives with no analyticity requirement (AD handles branches and non-smooth primitives by convention) and, in reverse mode, all $n$ components for the price of one.

**The hierarchy, in one line.**

$$
\text{finite differences } \bigl(\eta^{p/(p+k)}\bigr) \ \prec\ \text{complex step } \bigl(\varepsilon_M,\ O(n)\bigr) \ \prec\ \text{forward AD } \bigl(\varepsilon_M,\ O(n),\ \text{no analyticity}\bigr) \ \prec\ \text{reverse AD } \bigl(\varepsilon_M,\ O(1)\bigr)
$$

with the caveat that each step to the right demands more access to the source.

$$
\boxed{E_{\mathrm{CS}}(h) \le \tfrac{h^{2}}{6}\lvert f'''\rvert + c\,\varepsilon_M\lvert f'\rvert: \text{ monotone in } h, \text{ no cancellation, but analyticity is mandatory}}
$$

*Key takeaway:* The accuracy ceiling is a theorem about *subtracting real function values*; escaping it requires changing the information channel — a complex perturbation, a nilpotent, or the chain rule itself.